# TechOps Intelligence Platform
## Notebook 05 — Tabular Metrics Pipeline 
**Phase:** 2 — Document Processing  
**Goal:** Convert raw system metrics (CPU, memory, disk, network)
          into natural language summaries and embed into ChromaDB

### What This Notebook Does
1. Load Westermo server metrics (19 real servers)
2. Load Kaggle system performance metrics
3. Detect anomaly windows using z-score
4. Convert metric patterns to natural language
5. Embed and store in ChromaDB logs collection
6. Test retrieval for diagnosis agent queries

### Why Tabular Metrics Matter
When an incident fires, the diagnosis agent needs
to answer: what was happening to CPU, memory, and
disk at the time? This pipeline makes metric
patterns searchable using natural language.

In [1]:
import os
import re
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy import stats

from sentence_transformers import SentenceTransformer
import chromadb

PROJECT_ROOT = Path("C:/Users/sudha/techops-intelligence")
os.chdir(PROJECT_ROOT)

RAW_TABULAR = PROJECT_ROOT / "data/raw/tabular"
PROCESSED   = PROJECT_ROOT / "data/processed"
EMBEDDINGS  = PROJECT_ROOT / "data/embeddings"

print("Imports complete")
print(f"Project root: {PROJECT_ROOT}")

# Check tabular data files
print("\nTabular files found:")
for f in sorted(RAW_TABULAR.rglob("*.csv")):
    size_mb = f.stat().st_size / 1024 / 1024
    rel     = f.relative_to(RAW_TABULAR)
    print(f"  {str(rel):55} {size_mb:6.1f} MB")

Imports complete
Project root: C:\Users\sudha\techops-intelligence

Tabular files found:
  westermo\system-1.csv                                     19.3 MB
  westermo\system-10.csv                                    19.6 MB
  westermo\system-11.csv                                    18.8 MB
  westermo\system-12.csv                                    19.2 MB
  westermo\system-14.csv                                    18.5 MB
  westermo\system-15.csv                                    19.2 MB
  westermo\system-16.csv                                    19.3 MB
  westermo\system-17.csv                                    18.9 MB
  westermo\system-18.csv                                    19.6 MB
  westermo\system-19.csv                                    19.0 MB
  westermo\system-3.csv                                     19.2 MB
  westermo\system-4.csv                                     19.0 MB
  westermo\system-5.csv                                     19.6 MB
  westermo\system-7.csv    

## 1. Load Models and ChromaDB
Reuse embedding model from previous notebooks
Connect to existing ChromaDB logs collection

In [2]:
print("Loading embedding model...")
embedding_model = SentenceTransformer(
    'sentence-transformers/all-mpnet-base-v2',
    device='cpu'
)
print("Embedding model ready")

# Connect ChromaDB
chroma_path = str(EMBEDDINGS / "chroma_db")
client      = chromadb.PersistentClient(path=chroma_path)

logs_collection = client.get_or_create_collection(
    name     = "logs",
    metadata = {"hnsw:space": "cosine"}
)

print(f"ChromaDB connected")
print(f"Logs collection current count: {logs_collection.count()}")


def get_existing_ids(collection) -> set:
    if collection.count() == 0:
        return set()
    existing = collection.get(include=[])
    return set(existing['ids'])


def should_embed(doc_id: str, existing_ids: set) -> bool:
    return doc_id not in existing_ids


existing_ids = get_existing_ids(logs_collection)
print(f"Already embedded: {len(existing_ids)} documents")

Loading embedding model...
Embedding model ready
ChromaDB connected
Logs collection current count: 142240
Already embedded: 142240 documents


In [3]:
knowledge_base_collection = client.get_or_create_collection(
    name     = "knowledge_base",
    metadata = {"hnsw:space": "cosine"}
)

print("Collection ready:", knowledge_base_collection.name)
print("Documents in collection:", knowledge_base_collection.count())

Collection ready: knowledge_base
Documents in collection: 11298


In [4]:
# Create all three collections using your existing client
knowledge_base_collection = client.get_or_create_collection(
    name     = "knowledge_base",
    metadata = {"hnsw:space": "cosine"}
)

postmortems_collection = client.get_or_create_collection(
    name     = "postmortems",
    metadata = {"hnsw:space": "cosine"}
)

# Confirm all collections are ready
for col in [logs_collection, postmortems_collection, knowledge_base_collection]:
    print(f"  {col.name}: {col.count()} documents")

  logs: 142240 documents
  postmortems: 1284 documents
  knowledge_base: 11298 documents


In [5]:
from tqdm import tqdm

def embed_and_store(
    texts     : list,
    metadatas : list,
    ids       : list,
    collection,
    model,
    batch_size: int = 64
):
    total  = len(texts)
    stored = 0
    errors = 0

    for i in tqdm(range(0, total, batch_size), desc=f"Embedding -> {collection.name}"):
        batch_texts = texts[i:i+batch_size]
        batch_meta  = metadatas[i:i+batch_size]
        batch_ids   = ids[i:i+batch_size]

        valid = [
            (t, m, id_)
            for t, m, id_ in zip(batch_texts, batch_meta, batch_ids)
            if t and len(t.strip()) > 10
        ]
        if not valid:
            continue

        v_texts, v_meta, v_ids = zip(*valid)

        try:
            embeddings = model.encode(
                list(v_texts),
                show_progress_bar=False,
                batch_size=batch_size
            )
            collection.add(
                documents  = list(v_texts),
                embeddings = embeddings.tolist(),
                metadatas  = list(v_meta),
                ids        = list(v_ids)
            )
            stored += len(v_texts)
        except Exception as e:
            errors += 1
            print(f"  Batch error at {i}: {e}")

    return stored, errors

print("embed_and_store defined successfully")

embed_and_store defined successfully


## 2. Explore Westermo Data
Understand schema before processing
19 real server CSVs with 24 metrics each

In [6]:
westermo_path = RAW_TABULAR / "westermo"
csv_files     = sorted(westermo_path.glob("*.csv"))

print(f"Westermo server files: {len(csv_files)}")

# Load one file to understand schema
sample_df = pd.read_csv(csv_files[0])
print(f"\nSample file: {csv_files[0].name}")
print(f"Shape      : {sample_df.shape}")
print(f"Columns    : {sample_df.columns.tolist()}")
print(f"\nFirst 3 rows:")
print(sample_df.head(3))
print(f"\nData types:")
print(sample_df.dtypes)
print(f"\nMissing values:")
print(sample_df.isnull().sum())
print(f"\nBasic stats:")
print(sample_df.describe())

Westermo server files: 16

Sample file: system-1.csv
Shape      : (85749, 24)
Columns    : ['timestamp', 'load-1m', 'load-5m', 'load-15m', 'sys-mem-swap-total', 'sys-mem-swap-free', 'sys-mem-free', 'sys-mem-cache', 'sys-mem-buffered', 'sys-mem-available', 'sys-mem-total', 'sys-fork-rate', 'sys-interrupt-rate', 'sys-context-switch-rate', 'sys-thermal', 'disk-io-time', 'disk-bytes-read', 'disk-bytes-written', 'disk-io-read', 'disk-io-write', 'cpu-iowait', 'cpu-system', 'cpu-user', 'server-up']

First 3 rows:
   timestamp  load-1m  load-5m  load-15m  sys-mem-swap-total  \
0          0     0.22     0.18      0.18         16953372672   
1         30     0.26     0.19      0.18         16953372672   
2         60     0.16     0.17      0.18         16953372672   

   sys-mem-swap-free  sys-mem-free  sys-mem-cache  sys-mem-buffered  \
0        16953372672    2071302144    10307330048        1937584128   
1        16953372672    2072969216    10307371008        1937584128   
2        169533726

## 3. Anomaly Detection Functions
Z-score based anomaly detection on metric time series
Identifies windows where metrics behaved abnormally
Each anomaly window becomes a text document for RAG

In [7]:
def detect_anomaly_windows(
    df: pd.DataFrame,
    metric_cols: list,
    z_threshold: float = 2.5,
    window_size: int = 5,
    show_progress: bool = False
) -> list:
    anomaly_windows = []

    columns = (
        tqdm(metric_cols, desc="Checking metrics", leave=False)
        if show_progress else metric_cols
    )

    for col in columns:
        if col not in df.columns:
            continue

        values = pd.to_numeric(
            df[col], errors="coerce"
        ).astype("float64").to_numpy()

        valid_mask = np.isfinite(values)
        valid_values = values[valid_mask]

        if len(valid_values) < 10:
            continue

        std = valid_values.std()
        if not np.isfinite(std) or std == 0:
            continue

        z_scores = np.abs(
            (valid_values - valid_values.mean()) / std
        )
        is_anomaly = z_scores > z_threshold

        if not is_anomaly.any():
            continue

        # Original DataFrame row positions of anomalous values
        positions = np.flatnonzero(valid_mask)[is_anomaly]
        scores = z_scores[is_anomaly]

        # Find group boundaries without a Python loop per anomaly
        split_at = np.flatnonzero(
            np.diff(positions) > window_size
        ) + 1

        group_starts = np.r_[0, split_at]
        group_ends = np.r_[split_at - 1, len(positions) - 1]

        for start_i, end_i in zip(group_starts, group_ends):
            start_pos = int(positions[start_i])
            end_pos = int(positions[end_i])

            # Values across the full time window
            window_values = values[start_pos:end_pos + 1]
            window_values = window_values[np.isfinite(window_values)]

            if len(window_values) == 0:
                continue

            max_z = float(scores[start_i:end_i + 1].max())

            anomaly_windows.append({
                "metric": col,
                "start_idx": start_pos,
                "end_idx": end_pos,
                "duration": end_pos - start_pos + 1,
                "max_zscore": round(max_z, 2),
                "mean_value": round(float(window_values.mean()), 2),
                "max_value": round(float(window_values.max()), 2),
                "severity": (
                    "critical" if max_z > 4.0 else
                    "high" if max_z > 3.0 else
                    "medium"
                )
            })

    return anomaly_windows

# Correct unit map based on actual Westermo schema
WESTERMO_UNITS = {
    'cpu-user'               : ('CPU user usage',    'ratio', 100),
    'cpu-system'             : ('CPU system usage',  'ratio', 100),
    'cpu-iowait'             : ('CPU IO wait',       'ratio', 100),
    'load-1m'                : ('1min load average', 'load',  1),
    'load-5m'                : ('5min load average', 'load',  1),
    'load-15m'               : ('15min load average','load',  1),
    'sys-mem-free'           : ('free memory',       'bytes', 1),
    'sys-mem-available'      : ('available memory',  'bytes', 1),
    'sys-mem-swap-free'      : ('free swap',         'bytes', 1),
    'sys-mem-cache'          : ('memory cache',      'bytes', 1),
    'disk-bytes-read'        : ('disk read',         'bytes', 1),
    'disk-bytes-written'     : ('disk write',        'bytes', 1),
    'disk-io-time'           : ('disk IO time',      'ratio', 100),
    'sys-thermal'            : ('CPU temperature',   'celsius',1),
    'sys-context-switch-rate': ('context switches',  'per_sec',1),
    'sys-interrupt-rate'     : ('interrupt rate',    'per_sec',1),
}

def format_value(metric: str, value: float) -> str:
    """Format metric value with correct unit label."""
    if metric not in WESTERMO_UNITS:
        return f"{value:.3f}"

    label, unit_type, multiplier = WESTERMO_UNITS[metric]
    scaled = value * multiplier

    if unit_type == 'ratio':
        return f"{scaled:.1f}%"
    elif unit_type == 'bytes':
        if scaled > 1e9:
            return f"{scaled/1e9:.2f} GB"
        elif scaled > 1e6:
            return f"{scaled/1e6:.2f} MB"
        elif scaled > 1e3:
            return f"{scaled/1e3:.2f} KB"
        return f"{scaled:.0f} bytes"
    elif unit_type == 'celsius':
        return f"{scaled:.1f}C"
    else:
        return f"{scaled:.2f}"


def metrics_to_natural_language(
    anomaly_window : dict,
    server_name    : str,
    metric_units   : dict = None
) -> str:
    metric   = anomaly_window['metric']
    severity = anomaly_window['severity']
    max_val  = anomaly_window['max_value']
    mean_val = anomaly_window['mean_value']
    duration = anomaly_window['duration']
    max_z    = anomaly_window['max_zscore']

    label    = WESTERMO_UNITS.get(metric, (metric, '', 1))[0]
    fmt_max  = format_value(metric, max_val)
    fmt_mean = format_value(metric, mean_val)
    pattern  = _classify_pattern(metric, max_val, max_z)

    return (
        f"Server {server_name} showed a {severity} anomaly "
        f"in {label}. "
        f"Peak value was {fmt_max} with mean {fmt_mean} "
        f"during the anomaly. "
        f"Duration was {duration} intervals "
        f"({max_z:.1f} standard deviations from baseline). "
        f"Pattern suggests {pattern}."
    )


def _classify_pattern(metric: str, max_val: float, max_z: float) -> str:
    m = metric.lower()

    if 'cpu-user' in m:
        pct = max_val * 100
        if pct > 80:
            return "CPU saturation from runaway process or traffic spike"
        return "elevated CPU user load"

    if 'cpu-iowait' in m:
        pct = max_val * 100
        if pct > 30:
            return "severe IO wait indicating disk bottleneck or network storage issue"
        return "elevated IO wait possibly from disk activity"

    if 'cpu-system' in m:
        return "elevated kernel CPU usage possibly from interrupt handling"

    if 'load-1m' in m or 'load-5m' in m:
        if max_val > 2.0:
            return "system overload with more runnable processes than CPU cores"
        return "elevated load average indicating increased process activity"

    if 'swap-free' in m:
        gb = max_val / 1e9
        if gb < 1.0:
            return "swap exhaustion indicating memory pressure or memory leak"
        return "swap usage detected indicating memory pressure"

    if 'mem-free' in m or 'mem-available' in m:
        gb = max_val / 1e9
        if gb < 2.0:
            return "critically low free memory with OOM risk"
        return "reduced available memory"

    if 'disk-io-time' in m:
        pct = max_val * 100
        if pct > 80:
            return "disk IO saturation indicating storage bottleneck"
        return "elevated disk IO activity"

    if 'disk-bytes' in m:
        mb = max_val / 1e6
        if mb > 100:
            return "large disk transfer possibly from backup or data migration"
        return "disk write activity spike"

    if 'thermal' in m:
        if max_val > 4.0:
            return "CPU overheating risk requiring cooling check"
        return "elevated CPU temperature"

    return "resource anomaly requiring investigation"


# Test on sample data
print("Anomaly detection functions defined")
print("\nTesting on sample Westermo file...")

sample_df  = pd.read_csv(csv_files[0])
num_cols   = sample_df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns available: {len(num_cols)}")
print(f"Columns: {num_cols[:10]}")

anomalies = detect_anomaly_windows(sample_df, num_cols[:5])
print(f"\nAnomalies detected in first 5 columns: {len(anomalies)}")

if anomalies:
    sample_text = metrics_to_natural_language(
        anomalies[0],
        csv_files[0].stem
    )
    print(f"\nSample natural language output:")
    print(f"{sample_text}")

Anomaly detection functions defined

Testing on sample Westermo file...
Numeric columns available: 24
Columns: ['timestamp', 'load-1m', 'load-5m', 'load-15m', 'sys-mem-swap-total', 'sys-mem-swap-free', 'sys-mem-free', 'sys-mem-cache', 'sys-mem-buffered', 'sys-mem-available']

Anomalies detected in first 5 columns: 1378

Sample natural language output:
Server system-1 showed a critical anomaly in 1min load average. Peak value was 1.81 with mean 1.25 during the anomaly. Duration was 4 intervals (8.0 standard deviations from baseline). Pattern suggests elevated load average indicating increased process activity.


## 4. Process Westermo Server Metrics
19 real servers, 24 metrics each
Detect anomalies and convert to text
Check existing embeddings before processing

In [8]:
BATCH_SIZE = 32

def embed_and_store(
    texts     : list,
    metadatas : list,
    ids       : list,
    collection,
    model,
    batch_size: int = 32
) -> tuple:
    stored = 0
    errors = 0

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_meta  = metadatas[i:i+batch_size]
        batch_ids   = ids[i:i+batch_size]

        valid = [
            (t, m, id_)
            for t, m, id_ in zip(batch_texts, batch_meta, batch_ids)
            if t and len(t.strip()) > 20
        ]
        if not valid:
            continue

        v_texts, v_meta, v_ids = zip(*valid)

        try:
            embeddings = model.encode(
                list(v_texts),
                show_progress_bar=False
            )
            collection.upsert(
                documents  = list(v_texts),
                embeddings = embeddings.tolist(),
                metadatas  = list(v_meta),
                ids        = list(v_ids)
            )
            stored += len(v_texts)
        except Exception as e:
            errors += 1
            if errors <= 3:
                print(f"Batch error: {e}")

    return stored, errors


# Process all 19 Westermo servers
westermo_results = {}
existing_ids     = get_existing_ids(logs_collection)

print(f"Processing {len(csv_files)} Westermo server files...")
print(f"Already embedded: {len(existing_ids)} documents")

for csv_file in tqdm(csv_files, desc="Westermo servers"):
    server_name = csv_file.stem  # system-1, system-2 etc.

    df       = pd.read_csv(csv_file)
    num_cols = df.select_dtypes(
        include=[np.number]
    ).columns.tolist()

    if not num_cols:
        print(f"No numeric columns in {server_name}")
        continue

    # Detect anomalies
    anomalies = detect_anomaly_windows(df, num_cols)

    if not anomalies:
        westermo_results[server_name] = 0
        continue

    # Convert to natural language
    texts     = []
    ids       = []
    metadatas = []

    for idx, anomaly in enumerate(anomalies):
        doc_id = f"westermo_{server_name}_{anomaly['metric']}_{idx}"

        if not should_embed(doc_id, existing_ids):
            continue

        text = metrics_to_natural_language(anomaly, server_name)

        texts.append(text)
        ids.append(doc_id)
        metadatas.append({
            "source"    : "westermo_metrics",
            "server"    : server_name,
            "metric"    : anomaly['metric'],
            "severity"  : anomaly['severity'],
            "max_zscore": str(anomaly['max_zscore']),
            "doc_type"  : "metric_anomaly"
        })

    if texts:
        stored, errors = embed_and_store(
            texts, metadatas, ids,
            logs_collection,
            embedding_model,
            BATCH_SIZE
        )
        westermo_results[server_name] = stored
    else:
        westermo_results[server_name] = 0

print("\nWestermo processing complete")
print(f"\nAnomalies embedded per server:")
total_westermo = 0
for server, count in sorted(westermo_results.items()):
    if count > 0:
        print(f"  {server:15} : {count} anomaly windows")
    total_westermo += count

print(f"\nTotal Westermo anomalies stored: {total_westermo}")
print(f"Logs collection count: {logs_collection.count()}")

Processing 16 Westermo server files...
Already embedded: 142240 documents


Westermo servers: 100%|██████████| 16/16 [00:11<00:00,  1.37it/s]


Westermo processing complete

Anomalies embedded per server:

Total Westermo anomalies stored: 0
Logs collection count: 142240


In [9]:
import json

SYNTHETIC_LOG_PATTERNS = [
    # Database
    "ERROR db.connection - Connection refused on port 5432 after 3 retries",
    "FATAL db.pool - Connection pool exhausted max_connections=100 waiting=47",
    "ERROR db.replication - Replica lag 45s exceeds threshold 10s",
    "FATAL db.storage - Cannot extend relation disk full ENOSPC",
    "ERROR db.query - Query timeout after 30000ms table=orders rows_examined=8M",
    # Memory
    "FATAL jvm - OutOfMemoryError Java heap space used=3.9GB max=4GB",
    "ERROR jvm.gc - GC overhead limit exceeded 98pct time in garbage collection",
    "WARN system - Swap usage 89pct available_swap=2GB total_swap=16GB",
    "ERROR kernel - OOM killer activated process=payment-service pid=4821",
    # Network
    "ERROR dns - Resolution failed for payment-svc.internal NXDOMAIN",
    "ERROR network - Packet loss 34pct on eth0 retransmit_rate=high",
    "ERROR lb - All backend targets unhealthy dropping connections",
    "WARN ssl - Certificate expires in 7 days for api.company.com",
    # Kubernetes
    "ERROR k8s - Pod payment-7d9f CrashLoopBackOff restarts=24",
    "ERROR k8s - Node worker-03 NotReady kubelet stopped posting status",
    "ERROR k8s - OOMKilled container=api exit_code=137 limit=512Mi",
    "WARN etcd - Apply took 2.5s threshold=1s leader election at risk",
    # Storage
    "ERROR storage - Disk usage 98pct on /dev/sda1 write failed ENOSPC",
    "ERROR nfs - Mount failed for nas01:/data stale file handle",
    "WARN backup - pg_dump failed exit_code=1 disk full on backup volume",
    # Application
    "ERROR http - GET /api/orders timeout 30000ms circuit breaker open",
    "ERROR kafka - Consumer lag 2.4M messages topic=orders partition=0",
    "FATAL app - Segmentation fault core dumped signal=11 SIGSEGV",
    "ERROR deploy - Rollout failed health check timeout pods in CrashLoop",
    "WARN redis - Eviction rate 45000/sec maxmemory policy allkeys-lru",
    # Security
    "ERROR auth - Token validation service unreachable all logins failing",
    "WARN security - Anomalous traffic 4.2M req/s from 847 IPs WAF blocking",
    "ERROR vault - Secret rotation failed db-password credential stale",
    "ERROR iam - AccessDenied s3 PutObject role app insufficient permissions",
]

# Embed synthetic logs
texts     = SYNTHETIC_LOG_PATTERNS
ids       = [f"synthetic_log_{i}" for i in range(len(texts))]
metadatas = [
    {
        "source"  : "synthetic_it_ops_logs",
        "severity": "critical" if "FATAL" in t
                    else "high" if "ERROR" in t
                    else "medium",
        "doc_type": "log_pattern"
    }
    for t in texts
]

existing_ids = get_existing_ids(logs_collection)
new_texts, new_ids, new_meta = zip(*[
    (t, id_, m)
    for t, id_, m in zip(texts, ids, metadatas)
    if should_embed(id_, existing_ids)
]) if any(should_embed(id_, existing_ids)
          for id_ in ids) else ([], [], [])

if new_texts:
    embeddings = embedding_model.encode(list(new_texts))
    logs_collection.upsert(
        documents  = list(new_texts),
        embeddings = embeddings.tolist(),
        metadatas  = list(new_meta),
        ids        = list(new_ids)
    )
    print(f"Synthetic logs embedded: {len(new_texts)}")
else:
    print("Synthetic logs already embedded")

print(f"Logs collection: {logs_collection.count()}")

Synthetic logs already embedded
Logs collection: 142240


In [10]:
def logs_search(query, collection, model, n=3):
    query_emb = model.encode([query]).tolist()
    results   = collection.query(
        query_embeddings = query_emb,
        n_results        = n,
        include          = ['documents', 'metadatas', 'distances']
    )
    return results


test_queries = [
    "CPU usage spiked to 98 percent on server",
    "memory exhaustion swap usage critical",
    "disk space running out ENOSPC error",
    "database connection refused port 5432",
    "network packet loss high latency",
    "OutOfMemoryError java heap space"
]

print("Retrieval test on logs collection\n")
scores = []

for query in test_queries:
    results   = logs_search(
        query, logs_collection, embedding_model, n=2
    )
    docs      = results['documents'][0]
    distances = results['distances'][0]
    metas     = results['metadatas'][0]
    score     = 1 - distances[0]
    scores.append(score)

    print(f"Query  : {query}")
    print(f"Score  : {score:.3f}")
    print(f"Source : {metas[0].get('source', 'unknown')}")
    print(f"Text   : {docs[0][:120]}")
    print()

print(f"Score summary")
print(f"  Min : {min(scores):.3f}")
print(f"  Max : {max(scores):.3f}")
print(f"  Avg : {np.mean(scores):.3f}")

Retrieval test on logs collection

Query  : CPU usage spiked to 98 percent on server
Score  : 0.635
Source : westermo_metrics
Text   : Server system-9 showed a medium anomaly in CPU user usage. Peak value was 29.0% with mean 12.0% during the anomaly. Dura

Query  : memory exhaustion swap usage critical
Score  : 0.569
Source : westermo_metrics
Text   : Server system-10 showed a critical anomaly in available memory. Peak value was 32.39 GB with mean 32.05 GB during the an

Query  : disk space running out ENOSPC error
Score  : 0.657
Source : synthetic_it_ops_logs
Text   : FATAL db.storage - Cannot extend relation disk full ENOSPC

Query  : database connection refused port 5432
Score  : 0.863
Source : synthetic_it_ops_logs
Text   : ERROR db.connection - Connection refused on port 5432 after 3 retries

Query  : network packet loss high latency
Score  : 0.305
Source : westermo_metrics
Text   : Server system-1 showed a medium anomaly in context switches. Peak value was 4028.80 with mean 4028.

In [11]:
print("=" * 55)
print("NOTEBOOK 05 - TABULAR PIPELINE COMPLETE")
print("=" * 55)

print(f"\nLogs collection breakdown:")
print(f"  Westermo anomalies : {total_westermo}")

print(f"\nAvg retrieval score  : {np.mean(scores):.3f}")
print(f"Min retrieval score  : {min(scores):.3f}")
print(f"Max retrieval score  : {max(scores):.3f}")

print(f"\nFull ChromaDB state:")
all_collections = client.list_collections()
grand_total     = 0
for col in all_collections:
    c     = client.get_collection(col.name)
    count = c.count()
    grand_total += count
    print(f"  {col.name:20} : {count:,}")

print(f"  {'Total':20} : {grand_total:,}")

NOTEBOOK 05 - TABULAR PIPELINE COMPLETE

Logs collection breakdown:
  Westermo anomalies : 0

Avg retrieval score  : 0.650
Min retrieval score  : 0.305
Max retrieval score  : 0.873

Full ChromaDB state:
  logs                 : 142,240
  playbooks            : 174
  visuals              : 1,574
  incidents            : 20,776
  postmortems          : 1,284
  knowledge_base       : 11,298
  Total                : 177,346


In [12]:
import subprocess
result = subprocess.run(
    ["pip", "install", "-U", "langchain-ollama"],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else "")
print(result.stderr[-200:] if result.stderr else "")
print("Done")

4->langchain-core<2.0.0,>=1.2.21->langchain-ollama) (0.4.2)


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

Done


In [13]:
import sys
if "generate_synthetic_data" in sys.modules:
    del sys.modules["generate_synthetic_data"]

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import generate_synthetic_data as gsd

gsd.OLLAMA_MODEL     = "qwen2.5:3b-instruct-q4_K_M"
gsd.BATCH_SIZE       = 10
gsd.embedding_model  = embedding_model
gsd.PROJECT_ROOT     = PROJECT_ROOT
gsd.get_existing_ids = get_existing_ids
gsd.should_embed     = should_embed
gsd.embed_and_store  = embed_and_store

print("All injected. Model:", gsd.OLLAMA_MODEL)

All injected. Model: qwen2.5:3b-instruct-q4_K_M


In [14]:
llm = gsd.build_llm()
print("LLM ready")
gsd.task1_generate_incidents(llm, postmortems_collection, total=1000)
gsd.task2_generate_sre_qa(llm, knowledge_base_collection, total=500)
gsd.task3_generate_logs(llm, logs_collection, total=300)

LLM ready
Loaded 992 existing incidents from JSON
Task 1: Generating 8 more incidents in 1 batches
Processing batch 1 of 1


2026-08-22 18:17:12 [WARNING] Ollama call failed (attempt 1/2): [WinError 10061] No connection could be made because the target machine actively refused it -- retrying in 3 s
2026-08-22 18:17:17 [ERROR] Ollama call failed after 2 attempt(s): [WinError 10061] No connection could be made because the target machine actively refused it -- skipping batch
2026-08-22 18:17:17 [WARNING] Batch 1 returned no incidents, skipping


Total incidents in JSON: 992. Path: C:\Users\sudha\techops-intelligence\data\processed\synthetic_incidents.json
Embedding incidents into postmortems collection
Task 1: All incidents already embedded, nothing new to add
Loaded 470 existing SRE Q&A pairs from JSON
Task 2: Generating 30 more SRE Q&A pairs in 3 batches
Processing batch 1 of 3


2026-08-22 18:17:19 [WARNING] Ollama call failed (attempt 1/2): [WinError 10061] No connection could be made because the target machine actively refused it -- retrying in 3 s
2026-08-22 18:17:24 [ERROR] Ollama call failed after 2 attempt(s): [WinError 10061] No connection could be made because the target machine actively refused it -- skipping batch
2026-08-22 18:17:24 [WARNING] Batch 1 returned no Q&A pairs, skipping


Processing batch 2 of 3


...

Total SRE Q&A pairs in JSON: 470. Path: C:\Users\sudha\techops-intelligence\data\processed\sre_qa_pairs.json
Embedding SRE Q&A pairs into knowledge_base collection
Task 2: All SRE Q&A pairs already embedded, nothing new to add
Loaded 280 existing log patterns from JSON
Task 3: Generating 20 more log patterns in 2 batches
Processing batch 1 of 2


2026-08-22 18:17:40 [WARNING] Ollama call failed (attempt 1/2): [WinError 10061] No connection could be made because the target machine actively refused it -- retrying in 3 s
2026-08-22 18:17:45 [ERROR] Ollama call failed after 2 attempt(s): [WinError 10061] No connection could be made because the target machine actively refused it -- skipping batch
2026-08-22 18:17:45 [WARNING] Batch 1 returned no log lines, skipping


Processing batch 2 of 2


2026-08-22 18:17:47 [WARNING] Ollama call failed (attempt 1/2): [WinError 10061] No connection could be made because the target machine actively refused it -- retrying in 3 s
2026-08-22 18:17:52 [ERROR] Ollama call failed after 2 attempt(s): [WinError 10061] No connection could be made because the target machine actively refused it -- skipping batch
2026-08-22 18:17:52 [WARNING] Batch 2 returned no log lines, skipping


Total log patterns in JSON: 280. Backup: C:\Users\sudha\techops-intelligence\data\processed\synthetic_logs.json
Embedding log patterns into logs collection
Task 3: All log patterns already embedded, nothing new to add


In [17]:
# ---------------------------------------------------------------------------
# 1. ChromaDB State Summary
# ---------------------------------------------------------------------------
print("=" * 60)
print("CHROMADB STATE SUMMARY")
print("=" * 60)

all_collections = client.list_collections()
grand_total = 0

for col_info in all_collections:
    col   = client.get_collection(col_info.name)
    count = col.count()
    grand_total += count
    print(f"  {col_info.name:25} : {count:,} documents")

print(f"  {'TOTAL':25} : {grand_total:,} documents")

# ---------------------------------------------------------------------------
# 2. Breakdown by source - chunked to avoid SQL variable limit
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("BREAKDOWN BY SOURCE")
print("=" * 60)

CHUNK_SIZE = 5000

for col_info in all_collections:
    col   = client.get_collection(col_info.name)
    total = col.count()
    if total == 0:
        continue

    print(f"\n  Collection: {col_info.name} ({total:,} docs)")
    sources = {}
    offset  = 0

    while offset < total:
        chunk = col.get(
            limit   = CHUNK_SIZE,
            offset  = offset,
            include = ["metadatas"]
        )
        for meta in chunk["metadatas"]:
            src = meta.get("source", "unknown") if meta else "unknown"
            sources[src] = sources.get(src, 0) + 1
        offset += CHUNK_SIZE

    for src, cnt in sorted(sources.items(), key=lambda x: -x[1]):
        print(f"    {src:40} : {cnt:,}")

CHROMADB STATE SUMMARY
  logs                      : 142,240 documents
  playbooks                 : 174 documents
  visuals                   : 1,574 documents
  incidents                 : 20,776 documents
  postmortems               : 1,284 documents
  knowledge_base            : 11,298 documents
  TOTAL                     : 177,346 documents

BREAKDOWN BY SOURCE

  Collection: logs (142,240 docs)
    westermo_metrics                         : 141,922
    synthetic_it_ops_logs                    : 280
    loghub_linux                             : 38

  Collection: playbooks (174 docs)
    incident_playbook                        : 174

  Collection: visuals (1,574 docs)
    rvlcdip                                  : 1,280
    rvlcdip_ocr                              : 294

  Collection: incidents (20,776 docs)
    real_ai_incidents                        : 10,776
    cybersecurity                            : 10,000

  Collection: postmortems (1,284 docs)
    synthetic_incident   

In [18]:
print("=" * 60)
print("SYNTHETIC DATA LOCATIONS")
print("=" * 60)

checks = [
    (postmortems_collection,    "synthetic_incident_"),
    (knowledge_base_collection, "sre_qa_"),
    (logs_collection,           "synthetic_log_"),
]

for col, prefix in checks:
    sources = {}
    total   = col.count()
    offset  = 0

    while offset < total:
        chunk = col.get(
            limit   = 5000,
            offset  = offset,
            include = ["metadatas"]
        )
        for meta in chunk["metadatas"]:
            src = meta.get("source", "unknown") if meta else "unknown"
            sources[src] = sources.get(src, 0) + 1
        offset += 5000

    synthetic_count = sources.get(
        "synthetic_incident" if prefix == "synthetic_incident_"
        else "sre_qa" if prefix == "sre_qa_"
        else "synthetic_it_ops_logs", 0
    )

    print(f"\n  Collection : {col.name}")
    print(f"  Total docs : {total:,}")
    print(f"  Synthetic  : {synthetic_count:,}")
    print(f"  Sources    :")
    for src, cnt in sorted(sources.items(), key=lambda x: -x[1]):
        print(f"    {src:40} : {cnt:,}")

SYNTHETIC DATA LOCATIONS

  Collection : postmortems
  Total docs : 1,284
  Synthetic  : 992
  Sources    :
    synthetic_incident                       : 992
    danluu_postmortem                        : 234
    generated_postmortem                     : 58

  Collection : knowledge_base
  Total docs : 11,298
  Synthetic  : 470
  Sources    :
    aws_well_architected.pdf                 : 3,600
    building_secure_and_reliable_systems.pdf : 2,289
    site_reliability_engineering.pdf         : 2,050
    sre_workbook.pdf                         : 1,915
    aws_genai_lens.pdf                       : 974
    sre_qa                                   : 470

  Collection : logs
  Total docs : 142,240
  Synthetic  : 280
  Sources    :
    westermo_metrics                         : 141,922
    synthetic_it_ops_logs                    : 280
    loghub_linux                             : 38
